## Curso: MACHINE LEARNING CLASIFICACIÓN Y REGRESIÓN

## Módulo 2: Métodos de validación de información
## Unidad 2: Evaluación

🧩 Objetivo

- Probar distintos tipos de split
- Optimizar parámetros

## Importación de librerias

In [1]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold, cross_val_score

from sklearn.datasets import make_regression
from sklearn.datasets import load_wine

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

from sklearn.metrics import r2_score
from sklearn.metrics import accuracy_score

# Validación train_test_split

**train_test_split** es una función en Scikit-learn que te permite dividir un conjunto de datos en dos partes principales:

* **Conjunto de entrenamiento:** La parte de los datos que se usa para entrenar el modelo.

* **Conjunto de prueba:** La parte que se usa para evaluar el modelo, es decir, ver qué tan bien está funcionando en datos que no ha visto antes.

Con **train_test_split**, podemos elegir qué porcentaje de los datos se va al entrenamiento y qué porcentaje se va a la prueba. Normalmente, se usa 80% para entrenar y 20% para probar.

In [2]:
# Creamos un dataframe genérico
df = pd.DataFrame({
    'ID': range(1, 101),
    'Doble': range(2, 202, 2),
    'Target': [i * 3.5 for i in range(1, 101)]
})

df.head()

,ID,Doble,Target
0,1,2,3.5
1,2,4,7.0
2,3,6,10.5
3,4,8,14.0
4,5,10,17.5


In [3]:
# selección variables
X = df[['ID', 'Doble']]
y = df['Target']

In [4]:
# Dividimos en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
print("Datos de entrenamiento", X_train.shape)
print("Etiquetas de entrenamiento", y_train.shape, "\n")

print("Datos de prueba", X_test.shape)
print("Etiquetas de prueba", y_test.shape)

Datos de entrenamiento (80, 2)
Etiquetas de entrenamiento (80,) 

Datos de prueba (20, 2)
Etiquetas de prueba (20,)


**¿Cómo usar train_test_split para crear un conjunto de validación?**

Una forma de usar train_test_split para crear también un conjunto de validación es dividir los datos dos veces: una para crear conjuntos de entrenamiento y prueba, y otra vez para dividir el conjunto de entrenamiento en entrenamiento y validación.

**Paso a paso con train_test_split:**
*Dividir el conjunto de datos original en:*

* **Conjunto de entrenamiento** (por ejemplo, el 80% de los datos).

* **Conjunto de prueba** (por ejemplo, el 20% de los datos).

**Dividir el conjunto de entrenamiento en:**

* **Conjunto de entrenamiento más pequeño** (por ejemplo, el 60% de los datos originales).

* **Conjunto de validación**(por ejemplo, el 20% de los datos originales).


***Con esto, terminamos con tres partes:***

* **Conjunto de entrenamiento:** Usado para entrenar el modelo.

* **Conjunto de validación:** Usado para validar el modelo y ajustar hiperparámetros.

* **Conjunto de prueba:** Usado para evaluar el modelo final.

In [6]:
# Dividimos el conjunto de entrenamiento en conjunto de entrenamiento y validación
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42)

In [7]:
print("Datos de entrenamiento finales", X_train.shape)
print("Etiquetas de entrenamiento finales", y_train.shape, "\n")

print("Datos de validación", X_val.shape)
print("Etiquetas de validación", y_val.shape)

Datos de entrenamiento finales (60, 2)
Etiquetas de entrenamiento finales (60,) 

Datos de validación (20, 2)
Etiquetas de validación (20,)


# K-Fold

**¿Qué es K-Fold?**

K-Fold es una técnica que te ayuda a evaluar tu modelo de machine learning de manera más confiable y robusta. A diferencia de una simple división en entrenamiento y prueba, en K-Fold divides los datos en K partes iguales (o "folds") y entrenas y evalúas el modelo K veces con diferentes combinaciones de los datos.

**Ejemplo sencillo con K-Fold (K = 5):**

Imagina que tienes 100 estudiantes, y quieres entrenar un modelo para predecir sus calificaciones en un examen final basándote en cuántas horas estudiaron.

Dividimos los datos en 5 folds, cada uno con 20 estudiantes.

**Iteración 1:**

Entrenas el modelo con los datos de los folds 2, 3, 4 y 5.

Pruebas el modelo con los datos del fold 1.

**Iteración 2:**

Entrenas el modelo con los datos de los folds 1, 3, 4 y 5.

Pruebas el modelo con los datos del fold 2.

**Iteración 3:**

Entrenas el modelo con los datos de los folds 1, 2, 4 y 5.

Pruebas el modelo con los datos del fold 3.

**Iteración 4:**

Entrenas el modelo con los datos de los folds 1, 2, 3 y 5.

Pruebas el modelo con los datos del fold 4.

**Iteración 5:**

Entrenas el modelo con los datos de los folds 1, 2, 3 y 4.

Pruebas el modelo con los datos del fold 5.

Al final, tendrás 5 evaluaciones (una por cada fold) y las promedias para obtener una estimación general del rendimiento del modelo.

### Ejemplo básico

Creamos un conjunto de datos sintético

In [8]:
# Usamos make_regression para generar datos de ejemplo. Esto es útil
# cuando no tenemos datos reales a mano y queremos probar un modelo.

# n_samples: número de filas (observaciones)

# n_features: número de columnas (variables predictoras)

# noise: nivel de "ruido" o aleatoriedad en los datos

# random_state: semilla para que los datos generados sean siempre los mismos,
# lo que garantiza la reproducibilidad de los resultados.

X, y = make_regression(n_samples=100, n_features=2, noise=25, random_state=42)

In [9]:
# Aquí definimos el algoritmo que vamos a entrenar. En este caso,
# una Regresión Lineal, que es un modelo simple para predecir un valor continuo.

model = LinearRegression()

In [10]:
# Configurar la Validación Cruzada (K-Fold) ---

# KFold es una estrategia para dividir nuestros datos.

# n_splits=5: divide el conjunto de datos en 5 "folds" o partes.

# shuffle=True: mezcla los datos antes de dividirlos. Es muy importante para
#               evitar que el orden de los datos afecte el resultado.

# random_state=42: asegura que la mezcla (shuffle) sea la misma cada vez
#                  que se ejecuta el código, para obtener resultados consistentes.

kf = KFold(n_splits=5, shuffle=True, random_state=42)

Ejecutamos CV

In [11]:
# Parámetros:
# model: el modelo a evaluar. LinearRegression()

# X, y: los datos completos.

# cv=kf: la estrategia de validación cruzada que definimos antes KFold

# scoring='r2': la métrica que queremos calcular.

scores = cross_val_score(model, X, y, cv=kf, scoring='r2')

print(f"Scores R² de cada fold (automático): {scores}")
print(f"Promedio de los scores R² (automático): {scores.mean():.4f}")

Scores R² de cada fold (automático): [0.90471266 0.95481957 0.95310691 0.90923676 0.94174768]
Promedio de los scores R² (automático): 0.9327


# Optimización

Queremos construir un modelo que clasifique el tipo de vino a partir de sus propiedades químicas.

Variables ejemplo:

  alcohol

  magnesium

  flavanoids

  color_intensity
  
  proline

Variable objetivo:

tipo de vino (3 clases)

In [12]:
data = load_wine()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0


In [13]:
X.shape

(178, 13)

## Split de datos

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

Ahora tenemos:

Train → entrenamiento y optimización

Test → evaluación final

## Validación cruzada (Cross Validation)

Probemos un modelo simple:

In [15]:
modelo = RandomForestClassifier(random_state=42)

scores = cross_val_score(
    modelo,
    X_train,
    y_train,
    cv=5
)

print(scores)
print("Accuracy promedio:", scores.mean())

[1.         1.         0.92857143 0.96428571 1.        ]
Accuracy promedio: 0.9785714285714286


El modelo logra aproximadamente 98% de accuracy promedio usando validación cruzada.

Esto es mejor que evaluar con un solo split porque:

  reduce la varianza

  usa más datos para entrenamiento.

## Optimización de hiperparámetros

Ahora intentaremos mejorar el modelo.

Random Forest tiene hiperparámetros importantes:

n_estimators

max_depth

min_samples_split

max_features

El objetivo es encontrar la mejor combinación.

### Grid Search

Grid Search prueba todas las combinaciones posibles.

Definimos un espacio de búsqueda:

In [16]:
param_grid = {
    "n_estimators":[50,100,200],
    "max_depth":[None,3,5],
    "min_samples_split":[2,5,7],
    "max_features":["sqrt","log2"]
}

In [17]:
inicio = time.time()

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid.fit(X_train, y_train)

tiempo_grid = time.time() - inicio

Mejores parámetros

In [18]:
grid.best_params_

{'max_depth': None,
 'max_features': 'sqrt',
 'min_samples_split': 2,
 'n_estimators': 100}

Mejor score

In [19]:
grid.best_score_

np.float64(0.9785714285714286)

### Resultado Grid Search

In [20]:
print("GRID SEARCH")
print("Mejor accuracy CV:", grid.best_score_)
print("Mejores parámetros:", grid.best_params_)
print("Tiempo:", tiempo_grid)

GRID SEARCH
Mejor accuracy CV: 0.9785714285714286
Mejores parámetros: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 100}
Tiempo: 56.98701477050781


**Desventaja de Grid Search**

Si el dataset fuera grande, esto sería costoso.

### Random Search

Random Search explora combinaciones aleatorias del espacio de hiperparámetros.

Esto permite explorar muchos valores posibles con menos evaluaciones.

In [21]:
inicio = time.time()

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    random_state=42
)

random_search.fit(X_train, y_train)

tiempo_random = time.time() - inicio

In [22]:
random_search.best_params_

{'n_estimators': 100,
 'min_samples_split': 5,
 'max_features': 'log2',
 'max_depth': 5}

### Resultados Random

In [23]:
print("RANDOM SEARCH")
print("Mejor accuracy CV:", random_search.best_score_)
print("Mejores parámetros:", random_search.best_params_)
print("Tiempo:", tiempo_random)

RANDOM SEARCH
Mejor accuracy CV: 0.9785714285714286
Mejores parámetros: {'n_estimators': 100, 'min_samples_split': 5, 'max_features': 'log2', 'max_depth': 5}
Tiempo: 24.17009472846985


**Ventaja:**

Puede encontrar soluciones casi tan buenas como Grid Search con mucho menos costo computacional.

### Optimización Bayesiana

La optimización bayesiana busca hiperparámetros de forma inteligente, utilizando la información de iteraciones previas.

Una biblioteca muy usada es:

`scikit-optimize`

In [24]:
!pip install scikit-optimize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 1.2 MB/s eta 0:00:00


In [25]:
from skopt import BayesSearchCV

In [30]:
param_space = {
    "n_estimators": (50, 100, 200),
    "max_depth": (None, 3, 5),
    "min_samples_split": (2, 5, 7),
    "max_features": ("sqrt","log2")
}

inicio = time.time()

bayes = BayesSearchCV(
    RandomForestClassifier(random_state=42),
    param_space,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    random_state=42
)

bayes.fit(X_train, y_train)

tiempo_bayes = time.time() - inicio

In [31]:
bayes.best_params_

OrderedDict([('max_depth', 5),
             ('max_features', 'log2'),
             ('min_samples_split', 7),
             ('n_estimators', 100)])

## Resultado

In [32]:
print("BAYES SEARCH")
print("Mejor accuracy CV:", bayes.best_score_)
print("Mejores parámetros:", bayes.best_params_)
print("Tiempo:", tiempo_bayes)

BAYES SEARCH
Mejor accuracy CV: 0.9785714285714286
Mejores parámetros: OrderedDict({'max_depth': 5, 'max_features': 'log2', 'min_samples_split': 7, 'n_estimators': 100})
Tiempo: 39.07574462890625


### Comparación

In [33]:
resultados = pd.DataFrame({
    "Metodo":["Grid Search","Random Search","Bayesian Search"],
    "Accuracy CV":[
        grid.best_score_,
        random_search.best_score_,
        bayes.best_score_
    ],
    "Tiempo (segundos)":[
        tiempo_grid,
        tiempo_random,
        tiempo_bayes
    ]
})

resultados

,Metodo,Accuracy CV,Tiempo (segundos)
0,Grid Search,0.978571,56.987015
1,Random Search,0.978571,24.170095
2,Bayesian Search,0.978571,39.075745


## Evaluación final

Tomamos el mejor modelo.

In [35]:
best_model = random_search.best_estimator_

y_pred = best_model.predict(X_test)

accuracy_score(y_test, y_pred)

1.0

In [37]:
X_test.shape

(36, 13)

In [38]:
X_train.shape

(142, 13)